# Temporal Activity

Historical observation activity by hour, weekday, and month.

In [ ]:
# Load shared path, database, export, and report-header helpers.
%run pathutils.ipynb
%run database.ipynb
%run export.ipynb
%run report-header.ipynb

# Configure optional spreadsheet and chart exports without hard-coded paths.
export_outputs = False
export_folder = get_export_folder_path()


In [ ]:
# Display the standard report metadata before the report body.
report_metadata = display_report_header('Temporal Activity')


In [ ]:
# Load each temporal grain from its own SQL file.
hourly = query_data('tracker', construct_query('tracker', 'reports', 'temporal-by-hour.sql', {}))
weekday = query_data('tracker', construct_query('tracker', 'reports', 'temporal-by-weekday.sql', {}))
monthly = query_data('tracker', construct_query('tracker', 'reports', 'temporal-by-month.sql', {}))
heatmap_data = query_data('tracker', construct_query('tracker', 'reports', 'temporal-heatmap.sql', {}))
hourly


In [ ]:
# Compare observations and distinct aircraft across the main time grains.
fig, axes = plt.subplots(3, 1, figsize=(12, 12))
hourly.plot.bar(ax=axes[0], x='Hour', y=['Observations', 'Aircraft'], title='Activity by hour of day')
weekday.plot.bar(ax=axes[1], x='Weekday', y=['Observations', 'Aircraft'], title='Activity by weekday')
monthly.plot.line(ax=axes[2], x='Month', y=['Observations', 'Aircraft'], marker='o', title='Activity by month')
axes[2].tick_params(axis='x', rotation=45)
plt.tight_layout()
if export_outputs: export_chart(export_folder, 'temporal-activity', 'png')


In [ ]:
# Build a complete weekday-by-hour matrix so absent periods appear as zero.
heatmap = heatmap_data.pivot(index='Weekday Number', columns='Hour', values='Observations').reindex(index=range(7), columns=range(24), fill_value=0).fillna(0)
weekday_labels = ['Sunday', 'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday']

# Render a simple static heat map with labelled axes and a shared colour scale.
fig, axis = plt.subplots(figsize=(14, 5))
image = axis.imshow(heatmap, aspect='auto', cmap='Blues')
axis.set(title='Observation heat map', xlabel='Hour of day', ylabel='Weekday', xticks=range(24), yticks=range(7), yticklabels=weekday_labels)
fig.colorbar(image, ax=axis, label='Observations')
plt.tight_layout()
if export_outputs:
    export_chart(export_folder, 'temporal-heatmap', 'png')
    export_to_spreadsheet(export_folder, 'temporal-activity.xlsx', {'Hourly': hourly, 'Weekday': weekday, 'Monthly': monthly, 'Heatmap': heatmap_data})
